# Lezione 9 — Pratica: fondamenti di VQE

Scoprirai in prima persona perché l'ansatz è importante quanto l'Hamiltoniano: un circuito che funziona perfettamente per un Hamiltoniano può restare bloccato con un altro, se non gli concedi sufficiente libertà di rotazione.

## Traccia

Nel primo esempio della lezione teorica (Hamiltoniano di un singolo qubit) hai usato `H = 1 + 2 * cirq.Z(qubit)` con un ansatz formato da una sola rotazione `ry`, e l'ottimizzatore ha trovato senza problemi l'energia minima teorica (−1).

Ora cambia l'Hamiltoniano in `H = 1 + 2 * cirq.X(qubit)`. Prima di eseguire qualsiasi cosa, rifletti: l'ansatz attuale ha solo una rotazione `ry`, che non può mai portare il qubit su un autostato di X. Se lo esegui così com'è, l'ottimizzatore resterà bloccato lontano dal minimo reale.

Il tuo compito:

1. Aggiungi all'ansatz una porta `cirq.rz` (con un secondo parametro addestrabile), in modo che il circuito abbia una libertà di rotazione sufficiente verso qualsiasi asse
2. Aggiorna l'Hamiltoniano a `1 + 2 * cirq.X(qubit)`
3. Ottimizza di nuovo con Nelder-Mead e verifica che ora converga davvero all'energia minima teorica (−1)

**Suggerimenti**

- Ti servono due simboli di `sympy` invece di uno, ad esempio `sympy.symbols("angulo_y angulo_z")`
- L'ordine delle porte nell'ansatz conta: applica prima `cirq.ry` e poi `cirq.rz` sullo stesso qubit
- La funzione `energia(parametros)` ora riceve una lista di due valori, non di uno; `np.random.uniform(0, 2 * np.pi, 2)` ti dà un punto di partenza casuale con due parametri
- L'energia minima teorica di `H = 1 + 2*X` è la stessa di `H = 1 + 2*Z` (−1), perché X e Z hanno gli stessi autovalori (+1 e −1); cambia solo l'asse verso cui deve ruotare l'ansatz

In [ ]:
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
from scipy.optimize import minimize

qubit = cirq.GridQubit(0, 0)

# Ansatz original: una única rotación Y entrenable (insuficiente para H = 1 + 2*X)
angulo = sympy.symbols("angulo")
circuito_vqe = cirq.Circuit(cirq.ry(angulo).on(qubit))

# TODO: cambia el Hamiltoniano de "1 + 2 * cirq.Z(qubit)" a "1 + 2 * cirq.X(qubit)"
hamiltoniano = 1 + 2 * cirq.Z(qubit)

# TODO: añade un segundo símbolo entrenable (por ejemplo, "angulo_z") y reconstruye
# el circuito para que incluya, además de la rotación ry, una rotación rz sobre
# el mismo qubit

entrada_vacia = tfq.convert_to_tensor([cirq.Circuit()])
capa_entrada = tf.keras.layers.Input(shape=(), dtype=tf.dtypes.string)
inicializador = tf.keras.initializers.RandomUniform(0, 2 * np.pi)
salida = tfq.layers.PQC(circuito_vqe, hamiltoniano,
                         differentiator=tfq.differentiators.CentralDifference(),
                         initializer=inicializador)(capa_entrada)
modelo_vqe = tf.keras.models.Model(inputs=capa_entrada, outputs=salida)


def energia(parametros):
    modelo_vqe.set_weights(np.array([parametros]))
    return modelo_vqe(entrada_vacia).numpy()[0][0]


# TODO: el punto de partida aleatorio ahora necesita tantos valores como
# parámetros entrenables tenga tu ansatz (uno por cada símbolo de sympy)
resultado = minimize(energia, np.random.uniform(0, 2 * np.pi, 1), method="Nelder-Mead")
print("Energía mínima encontrada:", resultado["fun"])
print("Parámetros óptimos:", resultado["x"])

<details>
<summary>💡 Mostra soluzione</summary>

```python
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
from scipy.optimize import minimize

qubit = cirq.GridQubit(0, 0)

# Ansatz con due parametri addestrabili: prima ry, poi rz.
# La rz aggiunge la libertà di rotazione che mancava per raggiungere un autostato di X
angulo_y, angulo_z = sympy.symbols("angulo_y angulo_z")
circuito_vqe = cirq.Circuit(
    cirq.ry(angulo_y).on(qubit),
    cirq.rz(angulo_z).on(qubit),
)

# Hamiltoniano cambiato da Z a X: H = 1*I + 2*X, energia fondamentale teorica -1
hamiltoniano = 1 + 2 * cirq.X(qubit)

entrada_vacia = tfq.convert_to_tensor([cirq.Circuit()])
capa_entrada = tf.keras.layers.Input(shape=(), dtype=tf.dtypes.string)
inicializador = tf.keras.initializers.RandomUniform(0, 2 * np.pi)
salida = tfq.layers.PQC(circuito_vqe, hamiltoniano,
                         differentiator=tfq.differentiators.CentralDifference(),
                         initializer=inicializador)(capa_entrada)
modelo_vqe = tf.keras.models.Model(inputs=capa_entrada, outputs=salida)


def energia(parametros):
    modelo_vqe.set_weights(np.array([parametros]))
    return modelo_vqe(entrada_vacia).numpy()[0][0]


# Punto di partenza casuale con due parametri, uno per ogni simbolo dell'ansatz
resultado = minimize(energia, np.random.uniform(0, 2 * np.pi, 2), method="Nelder-Mead")
print("Energia minima trovata:", resultado["fun"])
print("Parametri ottimali:", resultado["x"])

# Con la rz aggiunta, il risultato converge a -1 (o molto vicino), la stessa energia
# fondamentale teorica dell'esempio originale con Z, perché X e Z condividono
# gli stessi autovalori (+1 e -1); ciò che cambiava era l'asse di rotazione necessario
```

</details>